# Module 02: Transfer Entropy Feature Selection for Time Series

## Learning Objectives

By completing this notebook, you will:
1. Compute transfer entropy (TE) between candidate feature time series and the target
2. Compare TE rankings with Granger causality rankings on the same data
3. Understand directed information flow — which features "cause" target movements
4. Apply surrogate significance testing to avoid selecting noise features
5. Visualise the directed information flow network across features

## Prerequisites
- Guide 02: Transfer entropy, Granger causality, and directed MI
- Notebook 01: Standard ITFS criteria
- Python: numpy, pandas, sklearn, matplotlib, statsmodels

## Estimated Time: 15 minutes

## Key Reference
Schreiber, T. (2000). Measuring information transfer. *Physical Review Letters*, 85(2), 461–464.

---

## 1. Setup and Data

We simulate a realistic financial time series scenario: a target return series and 8 candidate features including genuine leading indicators, lagged features with no directional causality, and pure noise. This gives us ground truth to validate our TE-based ranker.

**Why simulation?** Real financial data has unknown ground truth. Simulation lets us verify that TE correctly identifies causal features and ignores correlated-but-not-causal ones.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded.")

### 1.1 Generate Synthetic Financial Time Series

We construct a data-generating process with known causal structure:
- **x1**: Causal feature — directly drives y with 1-lag (like an inventory signal driving price)
- **x2**: Causal feature — drives y with 2-lag (like a macro indicator)
- **x3**: Non-causal but correlated — y partially drives x3 (reverse causality)
- **x4**: Non-causal but correlated via a common driver — spurious correlation
- **x5–x8**: Pure noise features independent of y

In [ ]:
T = 1000  # Time steps (~4 years of daily data)

# Common driver: a latent market factor (e.g., risk appetite)
market_factor = 0.5 * np.random.randn(T)

# x1: Genuine causal feature (lag-1)
# x1[t] causes y[t+1] — like an inventory signal predicting next-day price
x1 = 0.8 * np.random.randn(T)
x1_effect = np.zeros(T)
x1_effect[1:] = 0.5 * x1[:-1]  # Lag-1 effect

# x2: Genuine causal feature (lag-2)
# x2[t] causes y[t+2] — like a macro signal with a 2-day lead
x2 = 0.6 * np.random.randn(T) + 0.4 * market_factor
x2_effect = np.zeros(T)
x2_effect[2:] = 0.4 * x2[:-2]  # Lag-2 effect

# Target: y driven by x1 (lag-1), x2 (lag-2), market factor, and noise
y_noise = 0.3 * np.random.randn(T)
y = x1_effect + x2_effect + 0.3 * market_factor + y_noise

# x3: Non-causal — y partially drives x3 (reverse causality)
# Common in financial data: prices influence sentiment indicators
x3 = np.zeros(T)
x3[1:] = 0.6 * y[:-1] + 0.4 * np.random.randn(T - 1)

# x4: Non-causal — shares the market factor (spurious correlation)
x4 = 0.7 * market_factor + 0.3 * np.random.randn(T)

# x5-x8: Pure noise (AR(1) processes for realism)
x5 = np.zeros(T)
x6 = np.zeros(T)
x7 = np.zeros(T)
x8 = np.zeros(T)
for t in range(1, T):
    x5[t] = 0.5 * x5[t-1] + np.random.randn()
    x6[t] = 0.3 * x6[t-1] + np.random.randn()
    x7[t] = 0.7 * x7[t-1] + np.random.randn()
    x8[t] = -0.2 * x8[t-1] + np.random.randn()

# Assemble into a DataFrame
df = pd.DataFrame({
    'x1_causal_lag1':   x1,
    'x2_causal_lag2':   x2,
    'x3_reverse_cause': x3,
    'x4_common_driver': x4,
    'x5_noise_ar1':     x5,
    'x6_noise_ar1':     x6,
    'x7_noise_ar1':     x7,
    'x8_noise_ar1':     x8,
    'y_target':         y,
})

feature_cols = [c for c in df.columns if c != 'y_target']

# Ground truth: which features are genuine causes of y?
true_causal = {'x1_causal_lag1', 'x2_causal_lag2'}
true_non_causal = set(feature_cols) - true_causal

print(f"Time series length: {T} steps")
print(f"Features: {len(feature_cols)}")
print(f"  True causal: {sorted(true_causal)}")
print(f"  Non-causal:  {sorted(true_non_causal)}")
print(f"\nCorrelations with y (misleading — include non-causal features):")
for col in feature_cols:
    corr = df[col].corr(df['y_target'])
    print(f"  {col:<25}: {corr:+.3f}")

Notice that **correlation is misleading** here: x3 and x4 have non-zero correlations with y despite not being causal. Transfer entropy will correctly rank them lower than x1 and x2 by conditioning on y's own history.

---
## 2. Transfer Entropy Implementation

Transfer entropy measures the directed information flow from source $X$ to target $Y$:
$$T_{X \to Y} = I(Y_{t+1}; X_{t-\ell:t} \mid Y_{t-k:t})$$

The key: we condition on $Y$'s own past. This controls for $Y$'s autocorrelation and ensures TE measures only the **additional** information $X$ provides beyond $Y$'s own dynamics.

In [ ]:
def discretise_series(ts, n_bins=10):
    """
    Quantile-based discretisation of a time series to integers [0, n_bins-1].

    Equal-frequency binning ensures each bin has enough samples
    for reliable MI estimation.
    """
    quantiles = np.linspace(0, 100, n_bins + 1)
    bin_edges = np.percentile(ts, quantiles)
    # Adjust edges to ensure all values are captured
    bin_edges[0] -= 1e-10
    bin_edges[-1] += 1e-10
    return (np.digitize(ts, bin_edges) - 1).astype(int)


def encode_history(ts_disc, t_start, lag, T):
    """
    Encode lagged history of a discretised series as a single integer variable.

    Creates a joint variable representing (ts[t], ts[t-1], ..., ts[t-lag+1])
    encoded as a single integer.

    Parameters
    ----------
    ts_disc : array of int
        Discretised time series
    t_start : int
        First valid time index (t_start = max_lag)
    lag : int
        Number of lag steps to include
    T : int
        Length of original series

    Returns
    -------
    array of int
        Joint encoded history for times [t_start, T)
    """
    n_vals = int(np.max(ts_disc)) + 1
    result = np.zeros(T - t_start, dtype=int)
    for l in range(1, lag + 1):
        # ts[t-l] for t in [t_start, T)
        result = result * n_vals + ts_disc[t_start - l: T - l]
    return result


def transfer_entropy(x, y, k=1, ell=1, n_bins=10):
    """
    Compute transfer entropy T_{X -> Y}.

    T_{X->Y} = I(Y_{t+1}; X_{t-ell:t} | Y_{t-k:t})
             = I(Y_{t+1}; X_hist, Y_hist) - I(Y_{t+1}; Y_hist)

    Parameters
    ----------
    x : array of shape (T,)
        Source time series (candidate feature)
    y : array of shape (T,)
        Target time series
    k : int
        Markov order for Y history (how many lags of Y to condition on)
    ell : int
        Number of lags of X to include
    n_bins : int
        Bins for discretisation

    Returns
    -------
    float : Transfer entropy (non-negative)
    """
    x_d = discretise_series(x, n_bins)
    y_d = discretise_series(y, n_bins)

    max_lag = max(k, ell)
    T_len = len(y_d)

    # Y_{t+1}: future target
    y_future = y_d[max_lag:]

    # Y history: k lags of Y (controls for autocorrelation)
    y_hist = encode_history(y_d, max_lag, k, T_len)

    # X history: ell lags of X
    x_hist = encode_history(x_d, max_lag, ell, T_len)

    # T_{X->Y} = I(y_future; x_hist | y_hist)
    # Via chain rule: I(y_f; x_h | y_h) = I(y_f; x_h, y_h) - I(y_f; y_h)
    n_xh = int(np.max(x_hist)) + 1
    n_yh = int(np.max(y_hist)) + 1
    xy_hist = x_hist * n_yh + y_hist  # Joint encoding

    te = (mutual_info_score(y_future, xy_hist) -
          mutual_info_score(y_future, y_hist))

    return max(0.0, te)  # TE >= 0 by construction


# Quick test
te_x1_y = transfer_entropy(df['x1_causal_lag1'].values,
                            df['y_target'].values, k=1, ell=1)
te_x5_y = transfer_entropy(df['x5_noise_ar1'].values,
                            df['y_target'].values, k=1, ell=1)
print(f"T(x1 -> y): {te_x1_y:.4f}  [true causal, should be high]")
print(f"T(x5 -> y): {te_x5_y:.4f}  [pure noise, should be low]")

---
## 3. Surrogate Significance Testing

Raw TE values can be positive due to finite-sample bias even for independent series. Surrogate testing builds a null distribution by shuffling the source series (destroying its temporal order while preserving its marginal distribution), then computing TE on the shuffled version.

In [ ]:
def te_with_significance(
    x, y, k=1, ell=1, n_bins=10, n_surrogates=200, alpha=0.05, seed=42
):
    """
    Transfer entropy with surrogate significance testing.

    Null hypothesis: X does not Granger-cause Y (in the TE sense).
    Surrogates: randomly shuffle X (destroying temporal order,
    preserving marginal distribution).

    Parameters
    ----------
    x, y : array of shape (T,)
        Source and target time series
    k : int
        Markov order for Y history
    ell : int
        Lags of X to include
    n_bins : int
        Discretisation bins
    n_surrogates : int
        Number of shuffle surrogates
    alpha : float
        Significance level
    seed : int
        Random seed for reproducibility

    Returns
    -------
    dict with keys:
        te : float — observed TE
        p_value : float — fraction of surrogates >= te
        significant : bool — True if p_value < alpha
        threshold : float — 95th percentile of null distribution
        null_dist : array — surrogate TE values
    """
    te_obs = transfer_entropy(x, y, k=k, ell=ell, n_bins=n_bins)

    rng = np.random.default_rng(seed)
    null_tes = np.zeros(n_surrogates)

    for i in range(n_surrogates):
        # Shuffle destroys temporal order but preserves marginal distribution
        x_shuffled = rng.permutation(x)
        null_tes[i] = transfer_entropy(x_shuffled, y, k=k, ell=ell, n_bins=n_bins)

    p_value = float(np.mean(null_tes >= te_obs))
    threshold = float(np.percentile(null_tes, 100 * (1 - alpha)))

    return {
        'te': te_obs,
        'p_value': p_value,
        'significant': p_value < alpha,
        'threshold': threshold,
        'null_dist': null_tes,
    }


# Test significance for x1 (causal) and x5 (noise)
print("Running surrogate tests (200 surrogates per feature — ~20s)...")
result_x1 = te_with_significance(df['x1_causal_lag1'].values,
                                   df['y_target'].values, n_surrogates=200)
result_x5 = te_with_significance(df['x5_noise_ar1'].values,
                                   df['y_target'].values, n_surrogates=200)

print(f"\nx1 (causal): TE={result_x1['te']:.4f}, "
      f"p={result_x1['p_value']:.3f}, significant={result_x1['significant']}")
print(f"x5 (noise):  TE={result_x5['te']:.4f}, "
      f"p={result_x5['p_value']:.3f}, significant={result_x5['significant']}")

---
## 4. Compare TE Rankings with Granger Causality

Granger causality tests whether including $X$'s past improves a linear VAR model for $Y$. For linear Gaussian data, Granger causality and TE are equivalent. For non-linear data, they can disagree — and TE is more general.

We compare both on our dataset to see where they agree and where TE has an advantage.

In [ ]:
def granger_causality_score(x, y, lag=1):
    """
    Compute Granger causality F-statistic for X->Y.

    Tests H0: X does not Granger-cause Y.
    Uses the F-test from comparing restricted and unrestricted VAR models.

    Parameters
    ----------
    x : array of shape (T,)
        Source time series
    y : array of shape (T,)
        Target time series
    lag : int
        Number of lags in the VAR model

    Returns
    -------
    dict: f_stat, p_value, r2_restricted, r2_unrestricted
    """
    T_len = len(y)

    # Build lagged design matrices
    # Restricted model: y[t] ~ y[t-1], ..., y[t-lag]
    # Unrestricted model: y[t] ~ y[t-1], ..., y[t-lag], x[t-1], ..., x[t-lag]

    # Align series at t = lag to T-1
    y_future = y[lag:]
    n = len(y_future)

    y_lags = np.column_stack([y[lag-l:-l if l > 0 else T_len]
                              for l in range(1, lag+1)])
    x_lags = np.column_stack([x[lag-l:-l if l > 0 else T_len]
                              for l in range(1, lag+1)])

    # Restricted model (Y lags only)
    lr = LinearRegression(fit_intercept=True)
    lr.fit(y_lags, y_future)
    rss_r = np.sum((y_future - lr.predict(y_lags))**2)

    # Unrestricted model (Y lags + X lags)
    X_full = np.hstack([y_lags, x_lags])
    lu = LinearRegression(fit_intercept=True)
    lu.fit(X_full, y_future)
    rss_u = np.sum((y_future - lu.predict(X_full))**2)

    # F-statistic: (RSS_R - RSS_U) / lag  /  RSS_U / (n - 2*lag - 1)
    df1 = lag              # Numerator degrees of freedom
    df2 = n - 2*lag - 1    # Denominator degrees of freedom

    if rss_u < 1e-12 or df2 <= 0:
        return {'f_stat': 0.0, 'p_value': 1.0}

    f_stat = ((rss_r - rss_u) / df1) / (rss_u / df2)
    p_value = 1.0 - stats.f.cdf(f_stat, df1, df2)

    return {
        'f_stat': max(0.0, float(f_stat)),
        'p_value': float(p_value),
    }


# Compute both TE and Granger causality for all features
print(f"{'Feature':<25} {'TE':>8} {'TE sig':>8} {'GC F-stat':>10} {'GC p':>8} {'True cause':>12}")
print("-" * 75)

comparison = []
for col in feature_cols:
    x_series = df[col].values
    y_series = df['y_target'].values

    te_res = te_with_significance(x_series, y_series, n_surrogates=100, seed=0)
    gc_res = granger_causality_score(x_series, y_series, lag=2)

    is_causal = col in true_causal
    comparison.append({
        'feature': col,
        'te': te_res['te'],
        'te_significant': te_res['significant'],
        'gc_f': gc_res['f_stat'],
        'gc_p': gc_res['p_value'],
        'gc_significant': gc_res['p_value'] < 0.05,
        'true_causal': is_causal,
    })

    marker = 'CAUSAL' if is_causal else '------'
    sig_te = 'YES' if te_res['significant'] else 'no'
    print(f"{col:<25} {te_res['te']:>8.4f} {sig_te:>8} "
          f"{gc_res['f_stat']:>10.2f} {gc_res['p_value']:>8.3f} {marker:>12}")

comp_df = pd.DataFrame(comparison)

---
## 5. Directed Information Flow Network

We compute the full pairwise TE matrix between all features and the target to visualise the information flow structure. This reveals:
- Which features are genuine predictors of y (high $T_{x \to y}$)
- Which features are driven by y (high $T_{y \to x}$, reverse causality like x3)
- Which features are causally neutral (low TE in both directions)

In [ ]:
all_series_names = feature_cols + ['y_target']
all_series = [df[c].values for c in all_series_names]
n_series = len(all_series_names)

# Compute TE matrix: TE[i, j] = T(series_i -> series_j)
# We compute only the features-to-y column plus y-to-features row
# (full matrix would be 9x9 which is slower)
te_to_y = np.zeros(len(feature_cols))    # T(x_i -> y)
te_from_y = np.zeros(len(feature_cols))  # T(y -> x_i)

print("Computing directed TE (features <-> target)...")
for i, col in enumerate(feature_cols):
    x_s = df[col].values
    y_s = df['y_target'].values
    te_to_y[i] = transfer_entropy(x_s, y_s, k=2, ell=2)
    te_from_y[i] = transfer_entropy(y_s, x_s, k=2, ell=2)

# Create network-style summary
direction_df = pd.DataFrame({
    'feature': feature_cols,
    'te_to_y': te_to_y,
    'te_from_y': te_from_y,
    'net_flow': te_to_y - te_from_y,  # Positive = feature drives y
    'true_causal': [col in true_causal for col in feature_cols],
}).sort_values('net_flow', ascending=False)

print(f"\n{'Feature':<25} {'T(x->y)':>9} {'T(y->x)':>9} {'Net flow':>10} {'True cause':>12}")
print("-" * 70)
for _, row in direction_df.iterrows():
    causal = 'CAUSAL' if row['true_causal'] else '------'
    print(f"{row['feature']:<25} {row['te_to_y']:>9.4f} "
          f"{row['te_from_y']:>9.4f} {row['net_flow']:>10.4f} {causal:>12}")

---
## 6. Visualise the Information Flow Network

We create three visualisations:
1. TE vs. Granger causality F-statistic scatter — shows where they agree and disagree
2. Directed net information flow — positive = feature drives y, negative = y drives feature
3. Null distribution for the most and least significant features

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)

# ---- Panel 1: TE vs Granger Causality scatter ----
ax1 = fig.add_subplot(gs[0, 0:2])
colors = ['crimson' if tc else 'steelblue'
          for tc in comp_df['true_causal']]
ax1.scatter(comp_df['gc_f'], comp_df['te'],
            c=colors, s=120, zorder=3, edgecolors='white', linewidths=0.5)

for _, row in comp_df.iterrows():
    ax1.annotate(row['feature'].split('_')[0],
                 (row['gc_f'], row['te']),
                 textcoords='offset points', xytext=(5, 3), fontsize=8)

ax1.set_xlabel('Granger Causality F-Statistic', fontsize=11)
ax1.set_ylabel('Transfer Entropy (nats)', fontsize=11)
ax1.set_title('TE vs. Granger Causality Rankings', fontsize=12)
red_patch = mpatches.Patch(color='crimson', label='True causal feature')
blue_patch = mpatches.Patch(color='steelblue', label='Non-causal feature')
ax1.legend(handles=[red_patch, blue_patch], fontsize=9)

# ---- Panel 2: Directed net information flow ----
ax2 = fig.add_subplot(gs[0, 2])
bar_colors = ['crimson' if r['true_causal'] else 'steelblue'
              for _, r in direction_df.iterrows()]
short_names = [n.replace('_causal_lag1', '').replace('_causal_lag2', '')
               .replace('_reverse_cause', '').replace('_common_driver', '')
               .replace('_noise_ar1', '')
               for n in direction_df['feature']]
bars = ax2.barh(short_names, direction_df['net_flow'],
                color=bar_colors, alpha=0.8)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Net info flow T(x→y) − T(y→x)', fontsize=10)
ax2.set_title('Directed Information Flow', fontsize=12)
ax2.legend(handles=[red_patch, blue_patch], fontsize=8)

# ---- Panel 3: T(x->y) vs T(y->x) bubble plot ----
ax3 = fig.add_subplot(gs[1, 0:2])
for i, (col, to_y, from_y, is_causal) in enumerate(
    zip(direction_df['feature'], direction_df['te_to_y'],
        direction_df['te_from_y'], direction_df['true_causal'])
):
    color = 'crimson' if is_causal else 'steelblue'
    ax3.scatter(to_y, from_y, s=200, c=color, zorder=3,
               edgecolors='white', linewidths=0.5)
    ax3.annotate(col.split('_')[0], (to_y, from_y),
                textcoords='offset points', xytext=(5, 3), fontsize=9)

# Diagonal line: symmetric (neither drives the other)
lims = [0, max(direction_df['te_to_y'].max(), direction_df['te_from_y'].max()) * 1.1]
ax3.plot(lims, lims, 'k--', alpha=0.3, label='Symmetric (no direction)')
ax3.set_xlabel('T(feature → target)', fontsize=11)
ax3.set_ylabel('T(target → feature)', fontsize=11)
ax3.set_title('Asymmetric Information Flow\n'
              'Upper-left = feature drives target; '
              'Lower-right = target drives feature', fontsize=11)
ax3.legend(handles=[red_patch, blue_patch,
                     mpatches.Patch(color='none', label='-- = symmetric')],
           fontsize=9)
ax3.set_xlim(lims)
ax3.set_ylim(lims)

# ---- Panel 4: TE feature ranking ----
ax4 = fig.add_subplot(gs[1, 2])
sorted_by_te = comp_df.sort_values('te', ascending=True)
te_colors = ['crimson' if r else 'steelblue'
             for r in sorted_by_te['true_causal']]
short = [n.replace('_causal_lag1', '').replace('_causal_lag2', '')
           .replace('_reverse_cause', '').replace('_common_driver', '')
           .replace('_noise_ar1', '')
           for n in sorted_by_te['feature']]
ax4.barh(short, sorted_by_te['te'], color=te_colors, alpha=0.8)
ax4.set_xlabel('Transfer Entropy T(x→y)', fontsize=10)
ax4.set_title('TE Feature Ranking', fontsize=12)
ax4.legend(handles=[red_patch, blue_patch], fontsize=8)

plt.suptitle('Transfer Entropy Analysis: Directed Information Flow',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('transfer_entropy_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Build a TE-Based Feature Selector

We combine TE computation, surrogate significance testing, and directionality into a single reusable feature selector class.

In [ ]:
class TransferEntropySelector:
    """
    Feature selector based on transfer entropy.

    Ranks features by T(x_i -> y) and optionally filters out
    statistically insignificant features using surrogate testing.

    Parameters
    ----------
    k : int
        Number of lags for Y history (Markov order)
    ell : int
        Number of lags for X
    n_bins : int
        Discretisation bins
    n_surrogates : int
        Surrogates for significance testing (0 = skip testing)
    alpha : float
        Significance level
    """

    def __init__(self, k=1, ell=1, n_bins=10,
                 n_surrogates=200, alpha=0.05):
        self.k = k
        self.ell = ell
        self.n_bins = n_bins
        self.n_surrogates = n_surrogates
        self.alpha = alpha
        self.scores_ = None
        self.p_values_ = None
        self.feature_names_ = None

    def fit(self, X_df, y_series, feature_names=None):
        """
        Compute TE scores for all features.

        Parameters
        ----------
        X_df : DataFrame or array of shape (T, p)
            Feature time series
        y_series : array of shape (T,)
            Target time series
        feature_names : list of str, optional
            Feature names

        Returns
        -------
        self
        """
        if isinstance(X_df, pd.DataFrame):
            self.feature_names_ = list(X_df.columns)
            X_arr = X_df.values
        else:
            X_arr = np.asarray(X_df)
            p = X_arr.shape[1]
            self.feature_names_ = feature_names or [f'x{i}' for i in range(p)]

        p = X_arr.shape[1]
        self.scores_ = np.zeros(p)
        self.p_values_ = np.ones(p)

        y_arr = np.asarray(y_series)

        for i in range(p):
            if self.n_surrogates > 0:
                res = te_with_significance(
                    X_arr[:, i], y_arr,
                    k=self.k, ell=self.ell,
                    n_bins=self.n_bins,
                    n_surrogates=self.n_surrogates,
                    alpha=self.alpha,
                    seed=i
                )
                self.scores_[i] = res['te']
                self.p_values_[i] = res['p_value']
            else:
                self.scores_[i] = transfer_entropy(
                    X_arr[:, i], y_arr,
                    k=self.k, ell=self.ell, n_bins=self.n_bins
                )

        return self

    def get_top_k(self, k, significant_only=True):
        """
        Return indices and names of top-k features by TE.

        Parameters
        ----------
        k : int
            Number of features to return
        significant_only : bool
            If True, only consider statistically significant features

        Returns
        -------
        dict: indices, names, scores
        """
        mask = (self.p_values_ < self.alpha) if significant_only else np.ones(len(self.scores_), dtype=bool)
        filtered_idx = np.where(mask)[0]

        if len(filtered_idx) == 0:
            # Fall back to top-k by score if nothing is significant
            filtered_idx = np.arange(len(self.scores_))

        # Sort by TE score descending
        sorted_idx = filtered_idx[np.argsort(-self.scores_[filtered_idx])]
        top_k_idx = sorted_idx[:k]

        return {
            'indices': top_k_idx.tolist(),
            'names': [self.feature_names_[i] for i in top_k_idx],
            'scores': self.scores_[top_k_idx].tolist(),
        }

    def summary(self):
        """Print a ranked summary of all features."""
        order = np.argsort(-self.scores_)
        print(f"{'Rank':<5} {'Feature':<25} {'TE':>8} {'p-value':>9} {'Sig':>5}")
        print("-" * 55)
        for rank, i in enumerate(order, 1):
            sig = 'YES' if self.p_values_[i] < self.alpha else 'no'
            print(f"{rank:<5} {self.feature_names_[i]:<25} "
                  f"{self.scores_[i]:>8.4f} {self.p_values_[i]:>9.3f} {sig:>5}")


# Apply the selector
print("Fitting TransferEntropySelector (this takes ~30s with 200 surrogates)...")
selector = TransferEntropySelector(k=2, ell=2, n_bins=10,
                                    n_surrogates=200, alpha=0.05)
selector.fit(df[feature_cols], df['y_target'])

print("\nTransfer Entropy Feature Rankings:")
selector.summary()

top2 = selector.get_top_k(k=2, significant_only=True)
print(f"\nTop 2 significant features: {top2['names']}")
print(f"True causal features:        {sorted(true_causal)}")
print(f"Correctly identified:        {set(top2['names']) == true_causal}")

---
## 8. Exercise: Extend to Non-Linear Causal Detection

**Task:** Create a dataset where x1 causes y through a non-linear relationship:
$$y_t = \sin(x_{1, t-1}) + 0.3 \varepsilon_t$$

Run both transfer entropy and Granger causality (linear) on this dataset. Verify that TE detects the non-linear causality while Granger causality misses it (or gives a weak signal).

**Expected behaviour:** TE score for x1 should be high and significant. Granger F-statistic for x1 should be low (because linear regression cannot capture $\sin$).

In [ ]:
# YOUR CODE HERE
# --------------------------------------------------

# Step 1: Generate the non-linear causal dataset
T_nl = 1000
np.random.seed(99)

# x1_nl: uniform random values in [-pi, pi]
# x2_nl: pure noise (AR(1) for realism)
# y_nl: sin(x1_nl[t-1]) + noise

# TODO:
# x1_nl = ...
# x2_nl = ...
# y_nl = ...


# Step 2: Compute TE for x1_nl -> y_nl and x2_nl -> y_nl
# te_nl_x1 = transfer_entropy(x1_nl, y_nl, k=1, ell=1)
# te_nl_x2 = transfer_entropy(x2_nl, y_nl, k=1, ell=1)


# Step 3: Compute Granger causality for x1_nl -> y_nl and x2_nl -> y_nl
# gc_nl_x1 = granger_causality_score(x1_nl, y_nl, lag=1)
# gc_nl_x2 = granger_causality_score(x2_nl, y_nl, lag=1)


# Step 4: Print comparison — TE should detect x1, Granger may not
# print(f"Non-linear causal dataset (y = sin(x1[t-1]) + noise)")
# print(f"  TE  x1->y: {te_nl_x1:.4f}  |  GC F-stat x1->y: {gc_nl_x1['f_stat']:.2f}")
# print(f"  TE  x2->y: {te_nl_x2:.4f}  |  GC F-stat x2->y: {gc_nl_x2['f_stat']:.2f}")

In [ ]:
# AUTO-GRADED TEST
# --------------------------------------------------
def test_nonlinear_exercise():
    """Verify the non-linear causal dataset is set up correctly."""
    assert 'x1_nl' in dir() and 'y_nl' in dir(), \
        "Create x1_nl and y_nl variables"
    assert len(x1_nl) == T_nl, f"x1_nl should have {T_nl} observations"
    assert len(y_nl) == T_nl, f"y_nl should have {T_nl} observations"

    # Check that y_nl depends on x1_nl non-linearly
    corr_linear = np.corrcoef(x1_nl[:-1], y_nl[1:])[0, 1]
    corr_nonlinear = np.corrcoef(np.sin(x1_nl[:-1]), y_nl[1:])[0, 1]

    print(f"Linear correlation (x1, y_next): {corr_linear:.3f}")
    print(f"Non-linear correlation (sin(x1), y_next): {corr_nonlinear:.3f}")
    assert abs(corr_nonlinear) > abs(corr_linear), \
        "sin(x1) should be more correlated with y than x1 itself"

    print("Test passed: non-linear causal dataset correctly constructed.")


# Uncomment to run after implementing:
# test_nonlinear_exercise()

---
## 9. Summary

### Key Takeaways

1. **Transfer entropy detects direction** — $T_{X \to Y} \neq T_{Y \to X}$. Standard MI is symmetric and cannot distinguish causes from effects.

2. **Conditioning on Y's history is essential** — without it, TE conflates autocorrelation with causal flow. The formula $T_{X \to Y} = I(Y_{t+1}; X_{t-\ell:t} \mid Y_{t-k:t})$ is what makes TE a proper causal measure.

3. **Surrogate testing is not optional** — without significance testing, you will select features with spuriously high TE due to finite-sample bias.

4. **TE generalises Granger causality** — for linear Gaussian systems they are equivalent. For non-linear dynamics (common in financial data), TE detects relationships that Granger causality misses.

5. **The directed flow network reveals reverse causality** — features driven by y (like sentiment indicators driven by price) have high $T_{y \to x}$ and should not be used as predictors.

### What's Next

- **Exercise 01**: Implement CMIM from the CLM framework and compare Rényi MI rankings
- **Module 03**: Wrapper methods — replace MI with model cross-validation error
- **Module 07**: Full time series pipeline with proper walk-forward cross-validation

### References

- Schreiber, T. (2000). Measuring information transfer. *Physical Review Letters* 85(2).
- Granger, C. W. J. (1969). Investigating causal relations by econometric models. *Econometrica* 37(3).
- Kraskov, A. et al. (2004). Estimating mutual information. *Physical Review E* 69(6).